In [1]:
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import umap

/home/wollerf/miniforge3/envs/work/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dims = [16, 32, 64]
NUM_RUNS = 10
base_path = "../data/preprocessed/graph_based/MIMIC/with_target_embeddings"
nf_importances = pd.read_csv("feature_ranks_NF.csv", index_col=0)
aplasia_importances = pd.read_csv("feature_ranks_aplasia.csv", index_col=0)
label_df = pd.read_csv("d_labitems.csv")
label_dict = {f"{id}" : label for id, label in zip(label_df["itemid"], label_df["label"])}
label_dict["label_aplasia"] = "label_aplasia"
label_dict["label_nf"] = "label_nf"

In [3]:
result_dict = {"dim": [], "run": [], "corr": [], "target": [], "Type": []}
for dim in dims:
    for run in range(NUM_RUNS):
        emb_file = os.path.join(base_path, f"MIMIC_variables_{dim}_{run}.tsv")
        emb_df = pd.read_csv(emb_file, sep='\t', index_col=0)
        emb_df.index = emb_df.index.astype("str")
        # Compute pairwise cosine similarities from NF to all other variables.
        nf_var = emb_df.loc[["label_nf"]]
        others = emb_df.drop(index=["label_aplasia", "label_nf"])
        sims = cosine_similarity(nf_var, others)[0]
        nf_similarities = pd.Series(sims, index=others.index.map(label_dict), name="nf_sim")
        # Compute pairwise cosine similarities from Aplasia to all other vars.
        aplasia_var = emb_df.loc[["label_aplasia"]]
        others = emb_df.drop(index=["label_aplasia", "label_nf"])
        aplasia_sims = cosine_similarity(aplasia_var, others)[0]
        aplasia_similarities = pd.Series(aplasia_sims, index=others.index.map(label_dict), name="aplasia_sim")
        # Join with corresponding feature importances.
        nf_joint = nf_importances.join(nf_similarities, how="inner")
        aplasia_joint = aplasia_importances.join(aplasia_similarities, how="inner")
        nf_corr = nf_joint["imp_mean"].corr(nf_joint["nf_sim"], method="spearman")
        aplasia_corr = aplasia_joint["imp_mean"].corr(aplasia_joint["aplasia_sim"], method="spearman")
        result_dict["dim"].append(dim)
        result_dict["run"].append(run)
        result_dict["corr"].append(nf_corr)
        result_dict["Type"].append("Actual")
        result_dict["target"].append("NF")
        result_dict["dim"].append(dim)
        result_dict["run"].append(run)
        result_dict["corr"].append(aplasia_corr)
        result_dict["target"].append("Aplasia")
        result_dict["Type"].append("Actual")
        
        nf_random = []
        aplasia_random = []
        
        for i in range(100):
            nf_joint['imp_mean'] = nf_joint['imp_mean'].sample(frac=1, random_state=i).values
            aplasia_joint["imp_mean"] = aplasia_joint["imp_mean"].sample(frac=1, random_state=i).values
            
            nf_corr = nf_joint["imp_mean"].corr(nf_joint["nf_sim"], method="spearman")
            aplasia_corr = aplasia_joint["imp_mean"].corr(aplasia_joint["aplasia_sim"], method="spearman")
            
            nf_random.append(nf_corr)
            aplasia_random.append(aplasia_corr)
        
        result_dict["dim"].append(dim)
        result_dict["run"].append(run)
        result_dict["corr"].append(sum(nf_random)/len(nf_random))
        result_dict["Type"].append("Random")
        result_dict["target"].append("NF")
        result_dict["dim"].append(dim)
        result_dict["run"].append(run)
        result_dict["corr"].append(sum(aplasia_random)/len(aplasia_random))
        result_dict["target"].append("Aplasia")
        result_dict["Type"].append("Random")


In [4]:
result_df = pd.DataFrame(result_dict)
best_df = result_df[result_df["corr"]>0.6185]
best_df

,dim,run,corr,target,Type
41,32,0,0.650444,Aplasia,Actual


In [5]:
best_path = "../data/preprocessed/graph_based/MIMIC/with_target_embeddings/MIMIC_variables_32_0.tsv"
best_df = pd.read_csv(best_path, sep='\t', index_col=0)
best_df
best_df.index = best_df.index.map(label_dict)
best_df

,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9,...,dim_22,dim_23,dim_24,dim_25,dim_26,dim_27,dim_28,dim_29,dim_30,dim_31
Hematocrit,0.473224,-0.095450,-0.052723,-0.677402,0.268412,0.239824,0.258986,-0.086200,0.385171,-0.224067,...,-0.205258,-0.195724,-0.353598,0.199308,-0.292633,-0.155169,0.017905,-0.197001,0.190331,0.013846
Hemoglobin,-0.079684,0.034286,-0.100011,-0.282665,0.195719,0.232843,-0.113605,-0.043115,0.039005,-0.013343,...,-0.076037,0.162973,0.058672,0.244853,-0.152730,0.027772,0.052075,0.026862,0.096451,0.285820
MCH,0.247269,-0.127881,-0.083532,0.070213,0.073819,0.217738,0.338467,0.248389,0.102847,-0.077505,...,-0.115247,0.205844,0.217223,0.384470,0.189949,0.327763,0.043346,0.016521,0.097491,-0.205362
MCHC,0.412004,-0.113845,-0.091439,-0.291524,-0.066951,-0.095764,0.019668,-0.071918,0.088936,-0.054218,...,-0.286508,0.079825,-0.264375,-0.243555,-0.090724,0.036333,0.105026,0.206897,0.027812,-0.104039
MCV,0.399525,-0.312830,-0.119274,-0.263653,-0.116783,0.176823,-0.014160,0.022436,0.001072,-0.108836,...,-0.221766,-0.056905,0.113792,-0.102694,0.245094,0.051107,-0.038577,-0.246815,0.030795,0.064513
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Potassium, Urine",-0.459028,0.638058,-0.084842,0.432383,0.200526,-0.023135,-0.152342,0.044059,-0.099320,0.149396,...,-0.143861,0.015737,-0.510031,0.137939,0.175637,0.156020,-0.037305,0.008865,0.063867,0.019086
Triglycerides,-0.246639,0.465969,0.063488,-0.034718,0.149135,-0.143981,-0.345456,-0.164430,-0.405859,0.585385,...,-0.191033,-0.209542,0.457233,-0.121687,0.681279,-0.138784,0.060592,0.261258,-0.131886,-0.235102
ARCH-1,0.233531,0.097109,-0.125038,0.122351,0.282791,-0.034200,-0.101272,-0.085565,-0.094436,0.001321,...,-0.124865,-0.136365,-0.475351,0.221434,0.050834,-0.100851,0.063951,0.070570,0.002361,0.000145
label_nf,0.051871,-0.124939,0.630586,0.249066,0.018527,0.137597,0.301717,0.332982,-0.001703,-0.043481,...,0.068692,0.373118,-0.278489,-0.340548,-0.170528,0.133538,0.134032,0.108507,-0.073625,-0.907262


In [10]:
# Compute pairwise cosine similarities from NF to all other variables.
var = best_df.loc[["label_aplasia"]]
sims = cosine_similarity(var, best_df)[0]
aplasia_similarities = pd.Series(sims, index=best_df.index, name="cosine_sim")
aplasia_similarities

Hematocrit          0.544074
Hemoglobin          0.226480
MCH                 0.355841
MCHC                0.406429
MCV                 0.377295
                      ...   
Potassium, Urine   -0.442253
Triglycerides      -0.121396
ARCH-1              0.036089
label_nf           -0.208872
label_aplasia       1.000000
Name: cosine_sim, Length: 102, dtype: float64

In [11]:
# Cluster high-dimensional embeddings and then visualizes in 2D.
import igraph as ig
import leidenalg as la
import numpy as np
from sklearn.neighbors import kneighbors_graph

# --- Parameters ---
n_neighbors = int(0.05 * len(best_df))

## 1. Process POME Embeddings
pome_adj = kneighbors_graph(
    best_df.to_numpy(),
    n_neighbors=n_neighbors,
    mode='connectivity',
    include_self=False,
    metric='cosine', 
)

# Convert sparse matrix to igraph
sources, targets = pome_adj.nonzero()
pome_ig = ig.Graph(
    n=pome_adj.shape[0],
    edges=list(zip(sources, targets)),
    directed=False
)

# Run Leiden
pome_partition = la.find_partition(
    pome_ig,
    la.RBConfigurationVertexPartition,
    seed=42
)
pome_labels = np.array(pome_partition.membership)
pome_k = len(pome_partition)
print(pome_k)

6


In [12]:
tsne = TSNE(
    n_components=2, 
    random_state=42, 
    init='pca', 
)

vis_dims = tsne.fit_transform(best_df)
vis_df = pd.DataFrame(vis_dims, index=best_df.index, columns=["tSNE_1", "tSNE_2"])
vis_df["cluster_id"] = pome_labels
vis_df = vis_df.join(aplasia_importances, how="left")
vis_df = vis_df.join(aplasia_similarities, how="left")
vis_df

,tSNE_1,tSNE_2,cluster_id,imp_mean,cosine_sim
Hematocrit,-2.915424,-1.460778,4,1.62,0.544074
Hemoglobin,-1.897226,-0.443236,0,1.07,0.226480
MCH,-0.153721,2.622409,2,1.14,0.355841
MCHC,-1.021749,-1.035687,4,1.67,0.406429
MCV,-1.290897,0.177532,2,0.82,0.377295
...,...,...,...,...,...
"Potassium, Urine",3.073871,-2.828872,1,0.02,-0.442253
Triglycerides,2.398270,-4.545022,1,0.18,-0.121396
ARCH-1,1.914312,-1.016657,3,0.29,0.036089
label_nf,-0.600946,-3.308131,5,NaN,-0.208872


In [13]:
vis_df.to_csv("MIMIC_variable_visualization_results.csv", index=True)